# Baseline LLM-модель для оценки резюме

В этом ноутбуке показан прототип базовой модели для задачи:

> **Оценка соответствия резюме конкретной вакансии**

Подход:
- используем открытую LLM-модель (`google/flan-t5-small`) через `transformers`;
- задаём промпт, где модель играет роль HR-специалиста;
- парсим ответ и получаем числовой скор и текстовое объяснение;
- если LLM не даёт корректный ответ, используем **эвристический baseline**:
  считаем overlap ключевых слов между резюме и описанием вакансии.

Это демонстрирует:
- промпт-инжиниринг,
- простую LLM-инференс-пайплайн,
- baseline-оценку качества (на наборе тестовых пар).


In [1]:
!pip install -q transformers accelerate sentencepiece

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Берём компактную модель, чтобы она точно залетела в Colab
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Используется устройство:", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Используется устройство: cpu


функции промпта и парсинга ответа

In [3]:
import re
from dataclasses import dataclass
from typing import Dict, Any

# Простейшие стоп-слова (RU + EN), чтобы не учитывать "в", "и", "the" и т.п.
RUS_STOPWORDS = {
    "и", "в", "во", "на", "с", "со", "к", "от", "до", "за", "по", "из", "у", "о",
    "об", "обо", "для", "при", "без", "как", "что", "это", "так", "же", "или",
    "но", "а", "как", "бы", "ли", "же", "то", "быть", "есть", "нет", "год", "года",
    "лет", "опыт", "работы", "работа", "компания", "командой", "команды"
}

ENG_STOPWORDS = {
    "and", "or", "but", "the", "a", "an", "to", "of", "in", "on", "at", "from",
    "for", "with", "by", "as", "is", "are", "was", "were", "be", "been", "have",
    "has", "had", "do", "does", "did", "that", "this", "these", "those", "it",
    "its", "we", "you", "they", "he", "she", "i", "our", "their", "your"
}

GENERIC_STOPWORDS = RUS_STOPWORDS | ENG_STOPWORDS

# Набор ключевых тех-терминов для backend/DS (можно расширять)
TECH_KEYWORDS = {
    "python", "django", "flask", "fastapi",
    "rest", "api", "restapi", "grpc",
    "postgresql", "postgres", "mysql", "sqlite", "mongodb",
    "sql", "nosql",

    "docker", "kubernetes", "k8s",
    "git", "github", "gitlab",

    "redis", "rabbitmq", "kafka",

    "linux", "unix",

    "pandas", "numpy", "sklearn", "scikit", "scikit-learn",
    "matplotlib", "pytorch", "tensorflow",

    "backend", "backend-разработчик", "backend-разработка",
    "data", "etl", "pipeline", "pipelines",
}


def build_prompt(job_description: str, resume: str) -> str:
    """
    Промпт для модели: просим оценить соответствие резюме вакансии
    и вернуть ответ в формате SCORE/EXPLANATION.
    """
    return f"""
You are an experienced HR specialist in IT.

You will be given:
1) a job description
2) a candidate's resume

Your task:
- evaluate how well the candidate fits the job on a scale from 0 to 100
  (0 = not suitable at all, 100 = perfect fit)
- briefly explain your reasoning in 1–3 sentences
- do NOT invent facts that are not present in the resume

Job description: {job_description}

Candidate resume: {resume}

Answer STRICTLY in the following format (in Russian):

SCORE: <number from 0 to 100>
EXPLANATION: <short explanation in Russian>

No JSON, no additional text, no quotes, no markdown.
"""


@dataclass
class ResumeEvaluationResult:
    score: float
    explanation: str
    raw_text: str
    raw_parsed: Dict[str, Any]


def _generate_raw(prompt: str, max_new_tokens: int = 128) -> str:
    """
    Вызов LLM-модели через transformers.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded.strip()


def _parse_score_and_explanation(text: str) -> Dict[str, Any]:
    """
    Парсим ответ вида:

    SCORE: 85
    EXPLANATION: Кандидат хорошо подходит...

    Возвращаем словарь с полями score и explanation.
    """
    text = text.strip()

    # Если модель вообще ничего не вернула
    if not text:
        return {
            "score": 0,
            "explanation": "",
        }

    # Ищем SCORE
    score_match = re.search(r"SCORE:\s*([0-9]{1,3})", text, flags=re.IGNORECASE)
    if score_match:
        score_val = int(score_match.group(1))
        score_val = max(0, min(100, score_val))  # ограничим диапазоном 0..100
    else:
        score_val = 0

    # Ищем EXPLANATION
    expl_match = re.search(
        r"EXPLANATION:\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL
    )
    if expl_match:
        explanation = expl_match.group(1).strip()
    else:
        explanation = text  # если не нашли блок, берём весь текст

    return {
        "score": score_val,
        "explanation": explanation,
    }


def _tokenize_meaningful(text: str):
    """
    Токенизация с выкидыванием стоп-слов и очень коротких токенов.
    """
    tokens = re.findall(r"\w+", text.lower())
    return [
        t for t in tokens
        if t not in GENERIC_STOPWORDS and len(t) > 2
    ]


def _heuristic_score(job_description: str, resume: str) -> float:
    """
    Улучшенный эвристический скор:

    1) Выделяем "содержательные" слова (без стоп-слов).
    2) Делим их на:
       - общие meaningful токены,
       - ключевые тех-ключевые слова (из TECH_KEYWORDS).
    3) Считаем два оверлапа:
       - skill_overlap: пересечение TECH_KEYWORDS между вакансией и резюме,
       - token_overlap: пересечение всех meaningful токенов.
    4) Финальный скор = 70% skill_overlap + 30% token_overlap.
    """

    jd_tokens = set(_tokenize_meaningful(job_description))
    cv_tokens = set(_tokenize_meaningful(resume))

    if not jd_tokens:
        return 0.0

    # Тех-ключевые слова в вакансии и резюме
    jd_tech = jd_tokens & TECH_KEYWORDS
    cv_tech = cv_tokens & TECH_KEYWORDS

    if jd_tech:
        skill_overlap = len(jd_tech & cv_tech) / len(jd_tech)
    else:
        skill_overlap = 0.0

    # Общий meaningful overlap
    token_overlap = len(jd_tokens & cv_tokens) / len(jd_tokens)

    # Взвешиваем: скиллы важнее "общих слов"
    final_score = 100.0 * (0.7 * skill_overlap + 0.3 * token_overlap)
    return round(final_score, 1)


def evaluate_resume(job_description: str, resume: str) -> ResumeEvaluationResult:
    """
    Основная функция оценки:

    1. Формируем промпт и вызываем LLM.
    2. Пытаемся вытащить SCORE/EXPLANATION из ответа модели.
    3. Если LLM не дал корректный ответ (пусто, нет SCORE),
       используем улучшенный эвристический baseline по ключевым словам.
    """
    prompt = build_prompt(job_description, resume)
    raw_text = _generate_raw(prompt)
    parsed = _parse_score_and_explanation(raw_text)

    score = float(parsed.get("score", 0))
    explanation = str(parsed.get("explanation", "")).strip()

    # Признак того, что LLM "сломался" и не дал нормальный числовой ответ
    if (score == 0) and (len(raw_text.strip()) < 10 or "score" not in raw_text.lower()):
        score = _heuristic_score(job_description, resume)
        explanation = (
            "Оценка рассчитана автоматически на основе совпадения ключевых технологий "
            "и содержательных терминов между резюме и описанием вакансии. "
            "Бóльшее количество общих технологий (Python, фреймворки, БД, Docker и т.п.) "
            "даёт более высокий балл."
        )

    return ResumeEvaluationResult(
        score=score,
        explanation=explanation,
        raw_text=raw_text,
        raw_parsed=parsed,
    )


print("Функции для оценки резюме загружены (LLM + улучшенный эвристический baseline).")


Функции для оценки резюме загружены (LLM + улучшенный эвристический baseline).


Пример вакансии и резюме (можешь подставить свои)

In [4]:
job_description_example = """
Компания ищет Python backend-разработчика.

Обязанности:
- Разработка и поддержка REST API
- Работа с базой данных PostgreSQL
- Написание тестов и участие в code review

Требования:
- Опыт коммерческой разработки на Python от 2 лет
- Знание FastAPI или Django
- Опыт работы с PostgreSQL
- Понимание Docker, Git
"""

resume_example = """
Опыт работы: 3 года Python-разработчиком.

Разрабатывал REST API на FastAPI и Flask, интеграции с внешними сервисами.
Работал с PostgreSQL и Redis, писал SQL-запросы, настраивал миграции.

Использовал Docker и Docker Compose, Git — для командной работы.
Есть опыт написания unit-тестов и настройки CI/CD.
"""

result = evaluate_resume(job_description_example, resume_example)

print("=== Итоговая оценка соответствия (single example) ===")
print("Score:", result.score)
print("Explanation:", result.explanation)

print("\n=== Сырой ответ модели (для анализа поведения LLM) ===")
print(repr(result.raw_text))


=== Итоговая оценка соответствия (single example) ===
Score: 63.7
Explanation: Оценка рассчитана автоматически на основе совпадения ключевых технологий и содержательных терминов между резюме и описанием вакансии. Бóльшее количество общих технологий (Python, фреймворки, БД, Docker и т.п.) даёт более высокий балл.

=== Сырой ответ модели (для анализа поведения LLM) ===
'No JSON, no additional text, no quotes, no markdown.'


In [5]:
test_cases = [
    {
        "name": "Сильное совпадение: Python backend",
        "job_description": """
Компания ищет Python backend-разработчика.

Обязанности:
- Разработка и поддержка REST API
- Работа с базой данных PostgreSQL
- Написание тестов и участие в code review

Требования:
- Опыт коммерческой разработки на Python от 2 лет
- Знание FastAPI или Django
- Опыт работы с PostgreSQL
- Понимание Docker, Git
""",
        "resume": """
Опыт работы: 3 года Python backend-разработчиком.

Разрабатывал REST API на FastAPI и Django, интеграции с внешними сервисами.
Активно работал с PostgreSQL (сложные запросы, индексы, миграции Alembic).
Использовал Docker и Docker Compose, Git (GitFlow) в команде из 6 человек.
Писал unit- и интеграционные тесты, участвовал в code review.
"""
    },
    {
        "name": "Среднее совпадение: Python + немного другого стека",
        "job_description": """
Ищем Python разработчика в финтех-проект.

Требования:
- Python от 1 года
- Базовый опыт работы с веб-фреймворками (FastAPI, Flask или Django)
- База данных PostgreSQL или MySQL
- Будет плюсом опыт с очередями (RabbitMQ, Kafka)
""",
        "resume": """
Младший разработчик, 1.5 года опыта.

Писал скрипты на Python для автоматизации отчетов, немного работал с Flask.
Использовал SQLite и MySQL на учебных проектах.
Есть базовый опыт работы с Docker и Git.
С очередями сообщений пока не работал, но есть теоретическое понимание.
"""
    },
    {
        "name": "Слабое совпадение: фронтенд под backend-вакансию",
        "job_description": """
Компания ищет Python backend-разработчика.

Требования:
- Python, FastAPI или Django
- PostgreSQL
- Опыт построения REST API
""",
        "resume": """
Frontend-разработчик.

2 года опыта работы с JavaScript, TypeScript, React.
Верстал адаптивные интерфейсы, настраивал Webpack, работал с Redux.
Серверной разработки не делал, Python использовал только на уровне простых скриптов.
С базами данных опыта почти нет.
"""
    },
    {
        "name": "Почти не совпадает: графический дизайнер",
        "job_description": """
Ищем data engineer / backend-разработчика.

Требования:
- Python, SQL
- Работа с большими данными, ETL-пайплайны
- Опыт с Airflow, Kafka или Spark
""",
        "resume": """
Графический дизайнер.

5 лет опыта в полиграфии и digital.
Работаю в Figma, Photoshop, Illustrator.
Делаю бренд-айдентику, лендинги, презентации.
Опыт программирования отсутствует.
"""
    },
    {
        "name": "Другая доменная область: Data Scientist",
        "job_description": """
Компания ищет Data Scientist.

Требования:
- Python
- Опыт работы с библиотеками: pandas, numpy, scikit-learn
- Построение и валидация моделей машинного обучения
- Понимание метрик (accuracy, ROC-AUC, F1)
""",
        "resume": """
Data Scientist, 2 года опыта.

Работал с Python, pandas, numpy, scikit-learn, matplotlib.
Строил модели бинарной классификации (логистическая регрессия, random forest),
настраивал гиперпараметры, оценивал качество по ROC-AUC и F1.
Готовил фичи, чистил данные, работал с SQL.
Опыт деплоя моделей в прод ограничен.
"""
    },
]

print(f"Загружено тестовых кейсов: {len(test_cases)}")


Загружено тестовых кейсов: 5


In [6]:
import pandas as pd

rows = []

for case in test_cases:
    result = evaluate_resume(case["job_description"], case["resume"])
    score = result.score

    if score >= 70:
        bucket = "high_match"
    elif score >= 40:
        bucket = "medium_match"
    else:
        bucket = "low_match"

    rows.append({
        "case_name": case["name"],
        "score": score,
        "bucket": bucket,
        "explanation": result.explanation,
        "raw_llm_output": result.raw_text,
    })

df = pd.DataFrame(rows)
df


,case_name,score,bucket,explanation,raw_llm_output
0,Сильное совпадение: Python backend,82.7,high_match,Оценка рассчитана автоматически на основе совп...,0
1,Среднее совпадение: Python + немного другого с...,33.4,low_match,Оценка рассчитана автоматически на основе совп...,0
2,Слабое совпадение: фронтенд под backend-вакансию,12.7,low_match,Оценка рассчитана автоматически на основе совп...,0
3,Почти не совпадает: графический дизайнер,0.0,low_match,Оценка рассчитана автоматически на основе совп...,0
4,Другая доменная область: Data Scientist,85.0,high_match,Оценка рассчитана автоматически на основе совп...,0


In [7]:
for case in test_cases:
    print("==== Тест:", case["name"], "====")
    result = evaluate_resume(case["job_description"], case["resume"])
    print("Score:", result.score)
    print("Explanation:", result.explanation)
    print("Raw:", repr(result.raw_text))
    print()


==== Тест: Сильное совпадение: Python backend ====
Score: 82.7
Explanation: Оценка рассчитана автоматически на основе совпадения ключевых технологий и содержательных терминов между резюме и описанием вакансии. Бóльшее количество общих технологий (Python, фреймворки, БД, Docker и т.п.) даёт более высокий балл.
Raw: '0'

==== Тест: Среднее совпадение: Python + немного другого стека ====
Score: 33.4
Explanation: Оценка рассчитана автоматически на основе совпадения ключевых технологий и содержательных терминов между резюме и описанием вакансии. Бóльшее количество общих технологий (Python, фреймворки, БД, Docker и т.п.) даёт более высокий балл.
Raw: '0'

==== Тест: Слабое совпадение: фронтенд под backend-вакансию ====
Score: 12.7
Explanation: Оценка рассчитана автоматически на основе совпадения ключевых технологий и содержательных терминов между резюме и описанием вакансии. Бóльшее количество общих технологий (Python, фреймворки, БД, Docker и т.п.) даёт более высокий балл.
Raw: '0'

==== Те

In [8]:
print("Введите описание вакансии (или оставьте пустым для примера):")
jd = input().strip()
if not jd:
    jd = job_description_example

print("\nВведите текст резюме (или оставьте пустым для примера):")
cv = input().strip()
if not cv:
    cv = resume_example

res = evaluate_resume(jd, cv)

print("\n=== Результат для ваших данных ===")
print("Score:", res.score)
print("Explanation:", res.explanation)

print("\n=== Сырой ответ модели ===")
print(repr(res.raw_text))


Введите описание вакансии (или оставьте пустым для примера):
Мы ищем Python backend-разработчика в команду внутреннего HR-сервиса.  Обязанности: - Разработка и поддержка REST API на Python - Интеграция с внешними сервисами и базами данных - Оптимизация производительности и качества кода - Участие в код-ревью и обсуждении архитектурных решений  Требования: - Уверенное знание Python 3 - Опыт разработки backend-приложений (FastAPI или Django/DRF) - Опыт работы с PostgreSQL или другой реляционной БД - Понимание принципов REST, HTTP, авторизации - Умение работать с Git  Будет плюсом: - Опыт контейнеризации (Docker) - Опыт написания тестов (pytest)

Введите текст резюме (или оставьте пустым для примера):
Python backend-разработчик, 3 года опыта.  Опыт: — Разработка REST API для внутренней CRM-системы на FastAPI — Интеграция с платежными сервисами и сторонними API — Проектирование и оптимизация схемы БД в PostgreSQL — Написание unit- и интеграционных тестов (pytest) — Настройка CI/CD, деплой 